# Transfer Learning with CNNs — Guided Fashion-MNIST + CIFAR-10 Challenge

**References**
- Course CNN notebook: https://colab.research.google.com/github/serivan/DeepLearning/blob/master/PyTorch/12_deep_computer_vision_with_cnns.ipynb
- Previous live CNN workflow: Fashion-MNIST convolutional classifier in PyTorch
- TorchVision pretrained models: https://docs.pytorch.org/vision/main/models.html
- Optuna: https://optuna.readthedocs.io/

## Learning goals
By the end of this lab you should be able to:
1. explain the difference between **training from scratch**, **feature extraction**, and **fine-tuning**;
2. adapt a pretrained CNN to a new classification task;
3. decide which parameters should be trainable and assign different learning rates to different parts of a network;
4. design and train your own CNN on CIFAR-10;
5. compare your CNN with transfer learning using a pretrained model of your choice;
6. use **Optuna** to tune the most relevant hyperparameters systematically;
7. retrain, evaluate, save, and reload the final selected model.

> The notebook is divided into a short **guided live section on Fashion-MNIST** and an **independent CIFAR-10 challenge**. The test set must not be used for hyperparameter selection.


In [ ]:
import copy
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
pin_memory = device.type == "cuda"

print("PyTorch:", torch.__version__)
print("device:", device)


# Part A — Guided live: transfer learning on Fashion-MNIST

The previous CNN lab trained all convolutional filters **from scratch**. Here we keep Fashion-MNIST so that the dataset is familiar, but we change the learning strategy:

\[
\text{ImageNet pretrained CNN} \rightarrow \text{replace head} \rightarrow \text{freeze} \rightarrow \text{train head} \rightarrow \text{fine-tune selected layers}
\]

The purpose is not to obtain the best possible Fashion-MNIST score. Fashion-MNIST is intentionally very different from ImageNet; this makes **domain mismatch** visible and gives us something meaningful to discuss.


## A1 — Recall the CNN trained from scratch

A compact CNN previously studied on Fashion-MNIST can be written as one `nn.Sequential` pipeline. We will not retrain it here; use it only as a reference for the conceptual comparison.

**Discuss before running:**
- where are the features learned in this model?
- which parameters start from random initialization?
- if we move to a pretrained network, what changes?


In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, n_classes),
        )

    def forward(self, x):
        return self.net(x)

reference_cnn = SmallCNN()
print(reference_cnn)
print("parameters:", f"{count_parameters(reference_cnn)[0]:,}" if 'count_parameters' in globals() else "run the helper cell below")


## A2 — Training and evaluation helpers

We deliberately use **two separate functions**:
- `train_one_epoch()` updates parameters;
- `evaluate()` never computes gradients or updates parameters.

This keeps the distinction explicit during the live session.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    """Train for one epoch and return mean loss and accuracy."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate without updating model parameters."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(X)
        loss = criterion(logits, y)

        running_loss += loss.item() * X.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.numel()

    return running_loss / total, correct / total


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable


## A3 — Pretrained ResNet-18: input requirements

We use **ResNet-18 pretrained on ImageNet**. The pretrained weights expect an ImageNet-style RGB input and their associated preprocessing.

Fashion-MNIST images are grayscale, so we must first convert them to RGB.

**Predict before coding:**
1. why can we not simply use the original 28×28 grayscale tensor?
2. what happens to computational cost when a 28×28 image is resized to an ImageNet-style input?
3. should the original ImageNet 1000-class head be kept?


In [ ]:
weights = models.ResNet18_Weights.DEFAULT

transfer_transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert("RGB")),
    weights.transforms(),
])

fashion_train_full = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=transfer_transform
)
fashion_test = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=transfer_transform
)

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(60_000, generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

LIVE_FAST = True
if LIVE_FAST:
    train_idx_live = train_idx[:4_000]
    valid_idx_live = valid_idx[:1_000]
    test_idx_live = list(range(1_000))
else:
    train_idx_live = train_idx
    valid_idx_live = valid_idx
    test_idx_live = list(range(10_000))

fashion_train = Subset(fashion_train_full, train_idx_live)
fashion_valid = Subset(fashion_train_full, valid_idx_live)
fashion_test_live = Subset(fashion_test, test_idx_live)

TL_BATCH_SIZE = 64 if device.type == "cuda" else 32
fashion_train_loader = DataLoader(
    fashion_train, batch_size=TL_BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
fashion_valid_loader = DataLoader(
    fashion_valid, batch_size=TL_BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)
fashion_test_loader = DataLoader(
    fashion_test_live, batch_size=TL_BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

X_fashion, y_fashion = next(iter(fashion_train_loader))
print("transfer batch:", X_fashion.shape, y_fashion.shape)
print("range after normalization:", float(X_fashion.min()), float(X_fashion.max()))


## A4 — Turn ResNet-18 into a Fashion-MNIST feature extractor

Feature-extraction workflow:
1. load pretrained weights;
2. freeze pretrained parameters;
3. replace the original classification head;
4. train only the new head.

**Important:** freezing parameters saves gradient/update work, but the forward pass through the backbone is still required.


In [ ]:
transfer_model = models.resnet18(weights=weights)

# Freeze all pretrained parameters.
for p in transfer_model.parameters():
    p.requires_grad = False

# Replace the ImageNet head.
in_features = transfer_model.fc.in_features
transfer_model.fc = nn.Linear(in_features, 10)
transfer_model = transfer_model.to(device)

total_params, trainable_params = count_parameters(transfer_model)
print(transfer_model.fc)
print(f"total parameters:     {total_params:,}")
print(f"trainable parameters: {trainable_params:,}")
print(f"trainable fraction:   {trainable_params / total_params:.4%}")

with torch.no_grad():
    logits = transfer_model(X_fashion[:4].to(device))
print("logits shape:", logits.shape)


## A5 — Train only the new head

Use `CrossEntropyLoss`. The optimizer must initially receive **only the parameters of the new classification head**.

After the short run, compare training and validation accuracy. Do not expect ImageNet → Fashion-MNIST transfer to be perfect: the source and target domains are very different.


In [ ]:
criterion_tl = nn.CrossEntropyLoss()
optimizer_head = torch.optim.AdamW(
    transfer_model.fc.parameters(), lr=1e-3, weight_decay=1e-4
)

LIVE_EPOCHS = 1 if LIVE_FAST else 2
for epoch in range(LIVE_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        transfer_model, fashion_train_loader, criterion_tl, optimizer_head
    )
    val_loss, val_acc = evaluate(
        transfer_model, fashion_valid_loader, criterion_tl
    )
    print(
        f"epoch {epoch + 1:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"valid loss {val_loss:.4f} acc {val_acc:.3f}"
    )

fashion_test_loss, fashion_test_acc = evaluate(
    transfer_model, fashion_test_loader, criterion_tl
)
print(f"live demo test accuracy: {fashion_test_acc:.3f}")


## A6 — From feature extraction to partial fine-tuning

Now allow the last residual stage (`layer4`) to adapt.

Use **discriminative learning rates**:
- a small learning rate for pretrained `layer4`;
- a larger learning rate for the newly initialized `fc` head.

**Discuss:** why is a smaller learning rate usually safer for pretrained parameters?


In [ ]:
for p in transfer_model.layer4.parameters():
    p.requires_grad = True

optimizer_ft = torch.optim.AdamW([
    {"params": transfer_model.layer4.parameters(), "lr": 1e-5},
    {"params": transfer_model.fc.parameters(), "lr": 1e-3},
], weight_decay=1e-4)

_, trainable_after_unfreeze = count_parameters(transfer_model)
print(f"trainable after unfreezing layer4: {trainable_after_unfreeze:,}")

RUN_ONE_FINE_TUNING_EPOCH = False
if RUN_ONE_FINE_TUNING_EPOCH:
    train_loss, train_acc = train_one_epoch(
        transfer_model, fashion_train_loader, criterion_tl, optimizer_ft
    )
    val_loss, val_acc = evaluate(
        transfer_model, fashion_valid_loader, criterion_tl
    )
    print(
        f"fine-tune | train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"valid loss {val_loss:.4f} acc {val_acc:.3f}"
    )


### Live checkpoint
Be ready to explain these four statements in your own words:
1. **Training from scratch:** all useful visual filters must be learned from the target dataset.
2. **Feature extraction:** the pretrained representation is fixed and only the new head is trained.
3. **Fine-tuning:** some pretrained layers are allowed to adapt to the target domain.
4. **Domain similarity matters:** transfer is generally easier when source and target visual statistics/tasks are related.


# Part B — Independent challenge: CIFAR-10

Work individually or in a small group according to the course instructions.

You must complete three experiments:

1. **Your own CNN** — design and train a CNN from scratch on CIFAR-10.
2. **Transfer learning** — choose a pretrained TorchVision model and adapt it to CIFAR-10.
3. **Optuna tuning** — tune meaningful architecture/training hyperparameters of your own CNN, analyze the search, retrain the best configuration, evaluate it once on the test set, and save it.

## Required evidence in the notebook
For each experiment, keep the code, outputs, and a short interpretation. At minimum report:
- architecture / selected backbone;
- total and trainable parameter counts;
- validation accuracy;
- training time or qualitative computational-cost observation;
- design choices and what you learned from the result.

For Optuna also report:
- search space;
- best trial and best parameters;
- completed / pruned / early-stopped trials;
- parameter importance or another systematic analysis of hyperparameter behavior;
- final test accuracy;
- saved checkpoint path.


## B1 — CIFAR-10 data

CIFAR-10 contains RGB 32×32 images from 10 classes. We create separate training and validation dataset objects so that data augmentation is applied only to training samples.

The code below is intentionally mostly prepared. Inspect it and explain the role of `RandomCrop`, `RandomHorizontalFlip`, and normalization.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

cifar_train_aug = datasets.CIFAR10(
    root="data", train=True, download=True, transform=train_transform
)
cifar_train_eval = datasets.CIFAR10(
    root="data", train=True, download=True, transform=eval_transform
)
cifar_test = datasets.CIFAR10(
    root="data", train=False, download=True, transform=eval_transform
)

cifar_classes = cifar_train_aug.classes

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(cifar_train_aug), generator=g)
valid_idx = perm[:5_000].tolist()
train_idx = perm[5_000:].tolist()

cifar_train = Subset(cifar_train_aug, train_idx)
cifar_valid = Subset(cifar_train_eval, valid_idx)

BATCH_SIZE = 128
cifar_train_loader = DataLoader(
    cifar_train, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
cifar_valid_loader = DataLoader(
    cifar_valid, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)
cifar_test_loader = DataLoader(
    cifar_test, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

X, y = next(iter(cifar_train_loader))
print("batch:", X.shape, y.shape)
print("classes:", cifar_classes)


## B2 — Experiment 1: design your own CNN

Design a model rather than copying the Fashion-MNIST architecture unchanged.

### Minimum requirements
Your network must:
- accept `[B, 3, 32, 32]` inputs;
- contain at least two convolutional stages;
- reduce spatial resolution using pooling or stride;
- produce 10 logits;
- expose at least **three configurable choices** that will later be tunable by Optuna (for example base channels, dropout, BatchNorm, activation).

You may use `AdaptiveAvgPool2d` to avoid hard-coding a flattened spatial size.

### Before training
Write 3–5 lines explaining your architecture and why you chose it.


In [ ]:
class StudentCNN(nn.Module):
    """One valid reference design; students may build a different architecture."""
    def __init__(
        self,
        n_classes=10,
        base_channels=32,
        dropout=0.2,
        use_batchnorm=True,
        activation="relu",
    ):
        super().__init__()

        Act = nn.ReLU if activation == "relu" else nn.GELU

        def norm(channels):
            return nn.BatchNorm2d(channels) if use_batchnorm else nn.Identity()

        self.net = nn.Sequential(
            nn.Conv2d(3, base_channels, 3, padding=1),
            norm(base_channels),
            Act(),
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            norm(base_channels),
            Act(),
            nn.MaxPool2d(2),

            nn.Conv2d(base_channels, 2 * base_channels, 3, padding=1),
            norm(2 * base_channels),
            Act(),
            nn.Conv2d(2 * base_channels, 2 * base_channels, 3, padding=1),
            norm(2 * base_channels),
            Act(),
            nn.MaxPool2d(2),

            nn.Conv2d(2 * base_channels, 4 * base_channels, 3, padding=1),
            norm(4 * base_channels),
            Act(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(4 * base_channels, n_classes),
        )

    def forward(self, x):
        return self.net(x)


student_model = StudentCNN(
    base_channels=32, dropout=0.2, use_batchnorm=True, activation="relu"
).to(device)

with torch.no_grad():
    print("output shape:", student_model(X[:4].to(device)).shape)
print("parameters:", f"{count_parameters(student_model)[0]:,}")


In [ ]:
baseline_criterion = nn.CrossEntropyLoss()
baseline_optimizer = torch.optim.AdamW(
    student_model.parameters(), lr=1e-3, weight_decay=1e-4
)
BASELINE_EPOCHS = 3

t0 = time.perf_counter()
for epoch in range(BASELINE_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        student_model, cifar_train_loader, baseline_criterion, baseline_optimizer
    )
    val_loss, val_acc = evaluate(
        student_model, cifar_valid_loader, baseline_criterion
    )
    print(
        f"epoch {epoch + 1:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"valid loss {val_loss:.4f} acc {val_acc:.3f}"
    )

baseline_seconds = time.perf_counter() - t0
baseline_val_loss, baseline_val_acc = evaluate(
    student_model, cifar_valid_loader, baseline_criterion
)
print(f"baseline validation accuracy: {baseline_val_acc:.3f}")
print(f"elapsed: {baseline_seconds:.1f} s")


### B2 reflection
Answer briefly:
1. Which architectural choice had the largest effect on parameter count?
2. Where does the model lose spatial resolution?
3. What evidence suggests underfitting or overfitting?
4. What would you change before increasing the number of epochs?


## B3 — Experiment 2: transfer learning with a model of your choice

Choose **one pretrained CNN** from TorchVision. Recommended choices for this lab:

| Model | TorchVision builder | Classification head |
|---|---|---|
| ResNet-18 | `models.resnet18` | `model.fc` |
| EfficientNet-B0 | `models.efficientnet_b0` | `model.classifier[-1]` |
| MobileNetV3-Small | `models.mobilenet_v3_small` | `model.classifier[-1]` |

You may choose another TorchVision classification model, but you must identify correctly:
- its default pretrained weights;
- its required input preprocessing;
- its classification head;
- a sensible last block for partial fine-tuning.

### Required comparison
1. **Feature extraction:** freeze the pretrained backbone and train only the new head.
2. **Partial fine-tuning:** unfreeze the final block and use a lower learning rate for that block than for the head.

Compare the two approaches using **validation**, not test, accuracy.


In [ ]:
BACKBONE_NAME = "resnet18"  # Reference solution; students may choose another.


def build_transfer_model(name, n_classes=10):
    if name == "resnet18":
        weights = models.ResNet18_Weights.DEFAULT
        model = models.resnet18(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, n_classes)
        head = model.fc
        last_block = model.layer4

    elif name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
        head = model.classifier[-1]
        last_block = model.features[-1]

    elif name == "mobilenet_v3_small":
        weights = models.MobileNet_V3_Small_Weights.DEFAULT
        model = models.mobilenet_v3_small(weights=weights)
        for p in model.parameters():
            p.requires_grad = False
        model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, n_classes)
        head = model.classifier[-1]
        last_block = model.features[-1]

    else:
        raise ValueError(f"Unsupported reference backbone: {name}")

    return model.to(device), weights, head, last_block


transfer_model_cifar, transfer_weights, transfer_head, transfer_last_block = (
    build_transfer_model(BACKBONE_NAME, n_classes=10)
)

# Use the preprocessing attached to the selected pretrained weights.
tl_transform = transfer_weights.transforms()
cifar_tl_train_full = datasets.CIFAR10(
    root="data", train=True, download=True, transform=tl_transform
)
cifar_tl_test = datasets.CIFAR10(
    root="data", train=False, download=True, transform=tl_transform
)

cifar_tl_train = Subset(cifar_tl_train_full, train_idx)
cifar_tl_valid = Subset(cifar_tl_train_full, valid_idx)

TL_BATCH_SIZE = 64 if device.type == "cuda" else 32
cifar_tl_train_loader = DataLoader(
    cifar_tl_train, batch_size=TL_BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
cifar_tl_valid_loader = DataLoader(
    cifar_tl_valid, batch_size=TL_BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

X_tl, y_tl = next(iter(cifar_tl_train_loader))
print("transfer input batch:", X_tl.shape)
print("total/trainable:", count_parameters(transfer_model_cifar))


In [ ]:
criterion_transfer = nn.CrossEntropyLoss()
optimizer_transfer_head = torch.optim.AdamW(
    transfer_head.parameters(), lr=1e-3, weight_decay=1e-4
)

TRANSFER_HEAD_EPOCHS = 2
for epoch in range(TRANSFER_HEAD_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        transfer_model_cifar, cifar_tl_train_loader,
        criterion_transfer, optimizer_transfer_head
    )
    val_loss, val_acc = evaluate(
        transfer_model_cifar, cifar_tl_valid_loader, criterion_transfer
    )
    print(
        f"feature extraction {epoch + 1:02d} | "
        f"train acc {train_acc:.3f} | valid acc {val_acc:.3f}"
    )

feature_val_loss, feature_val_acc = evaluate(
    transfer_model_cifar, cifar_tl_valid_loader, criterion_transfer
)
_, feature_trainable = count_parameters(transfer_model_cifar)
print("feature-extraction trainable params:", f"{feature_trainable:,}")
print("feature-extraction val acc:", f"{feature_val_acc:.3f}")

# Partial fine-tuning.
for p in transfer_last_block.parameters():
    p.requires_grad = True

optimizer_partial_ft = torch.optim.AdamW([
    {"params": transfer_last_block.parameters(), "lr": 1e-5},
    {"params": transfer_head.parameters(), "lr": 5e-4},
], weight_decay=1e-4)

PARTIAL_FT_EPOCHS = 1
for epoch in range(PARTIAL_FT_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        transfer_model_cifar, cifar_tl_train_loader,
        criterion_transfer, optimizer_partial_ft
    )
    val_loss, val_acc = evaluate(
        transfer_model_cifar, cifar_tl_valid_loader, criterion_transfer
    )
    print(
        f"partial fine-tune {epoch + 1:02d} | "
        f"train acc {train_acc:.3f} | valid acc {val_acc:.3f}"
    )

partial_ft_val_loss, partial_ft_val_acc = evaluate(
    transfer_model_cifar, cifar_tl_valid_loader, criterion_transfer
)
_, partial_ft_trainable = count_parameters(transfer_model_cifar)
print("partial-FT trainable params:", f"{partial_ft_trainable:,}")
print("partial-FT val acc:", f"{partial_ft_val_acc:.3f}")


### B3 reflection
Discuss:
1. Why is ImageNet → CIFAR-10 transfer conceptually more natural than ImageNet → Fashion-MNIST?
2. Did feature extraction or partial fine-tuning work better in your run?
3. How much did the pretrained preprocessing increase input size and computational cost?
4. Is the model with the highest validation accuracy necessarily the best engineering choice? Consider memory, latency, and parameter count.


## B4 — Experiment 3: systematic hyperparameter tuning with Optuna

Tune **your StudentCNN**, not the test set.

To keep the search computationally manageable:
- use a fixed subset of the training and validation data during the Optuna study;
- keep each trial short;
- use both **Optuna pruning** and **per-trial early stopping**;
- retrain the best configuration after the search.

### Minimum search space
Include at least:
- one model-capacity parameter (e.g. `base_channels`);
- one regularization parameter (`dropout`, BatchNorm, or weight decay);
- activation or another architecture choice;
- optimizer;
- learning rate;
- weight decay;
- momentum when SGD is selected.

You may add parameters, but remember that larger search spaces require more trials.


In [ ]:
# Install Optuna in Colab if necessary.
try:
    import optuna
except ImportError:
    %pip -q install optuna
    import optuna

from optuna.importance import get_param_importances


In [ ]:
# Fixed subsets for faster Optuna trials.
SEARCH_TRAIN_N = 12_000
SEARCH_VALID_N = 3_000

search_train = Subset(cifar_train_aug, train_idx[:SEARCH_TRAIN_N])
search_valid = Subset(cifar_train_eval, valid_idx[:SEARCH_VALID_N])

search_train_loader = DataLoader(
    search_train, batch_size=BATCH_SIZE, shuffle=True,
    pin_memory=pin_memory, num_workers=2
)
search_valid_loader = DataLoader(
    search_valid, batch_size=BATCH_SIZE, shuffle=False,
    pin_memory=pin_memory, num_workers=2
)

SEARCH_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_MIN_DELTA = 1e-3
N_TRIALS = 12


In [ ]:
def build_student_model_from_config(config):
    return StudentCNN(
        n_classes=10,
        base_channels=int(config["base_channels"]),
        dropout=float(config["dropout"]),
        use_batchnorm=bool(config["use_batchnorm"]),
        activation=config["activation"],
    ).to(device)


def make_optimizer(model, config):
    if config["optimizer"] == "adamw":
        return torch.optim.AdamW(
            model.parameters(),
            lr=float(config["lr"]),
            weight_decay=float(config["weight_decay"]),
        )
    elif config["optimizer"] == "sgd":
        return torch.optim.SGD(
            model.parameters(),
            lr=float(config["lr"]),
            momentum=float(config.get("momentum", 0.0)),
            weight_decay=float(config["weight_decay"]),
        )
    else:
        raise ValueError(config["optimizer"])


def objective(trial):
    config = {
        "base_channels": trial.suggest_categorical(
            "base_channels", [16, 32, 48, 64]
        ),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5, step=0.1),
        "use_batchnorm": trial.suggest_categorical(
            "use_batchnorm", [True, False]
        ),
        "activation": trial.suggest_categorical(
            "activation", ["relu", "gelu"]
        ),
        "optimizer": trial.suggest_categorical(
            "optimizer", ["adamw", "sgd"]
        ),
        "lr": trial.suggest_float("lr", 1e-4, 3e-2, log=True),
        "weight_decay": trial.suggest_float(
            "weight_decay", 1e-6, 1e-3, log=True
        ),
    }

    if config["optimizer"] == "sgd":
        config["momentum"] = trial.suggest_float(
            "momentum", 0.0, 0.95, step=0.05
        )

    model = build_student_model_from_config(config)
    criterion = nn.CrossEntropyLoss()
    optimizer = make_optimizer(model, config)

    best_val_acc = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    early_stopped = False

    for epoch in range(SEARCH_EPOCHS):
        train_one_epoch(model, search_train_loader, criterion, optimizer)
        _, val_acc = evaluate(model, search_valid_loader, criterion)

        if val_acc > best_val_acc + EARLY_STOPPING_MIN_DELTA:
            best_val_acc = val_acc
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        trial.report(val_acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            early_stopped = True
            break

    trial.set_user_attr("best_epoch", best_epoch)
    trial.set_user_attr("early_stopped", early_stopped)
    return best_val_acc


In [ ]:
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=4,
    n_warmup_steps=1,
)
study = optuna.create_study(direction="maximize", pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS)

print("best validation accuracy:", study.best_value)
print("best parameters:", study.best_params)


## B5 — Analyze the Optuna study

Do not stop at `study.best_params`. Analyze **behavior**, not just the winner.

Required analysis:
1. show the best trials in a DataFrame;
2. count completed and pruned trials;
3. count trials that ended through your own early stopping logic;
4. compute parameter importance;
5. discuss at least **two relationships** between hyperparameters and validation performance.

Examples of questions:
- does a larger model always help?
- is strong dropout useful for every channel width?
- does the best learning-rate region depend on optimizer choice?
- does BatchNorm change the useful learning-rate range?


In [ ]:
trials_df = study.trials_dataframe()
display(
    trials_df.sort_values("value", ascending=False).head(10)
)

completed = sum(t.state == optuna.trial.TrialState.COMPLETE for t in study.trials)
pruned = sum(t.state == optuna.trial.TrialState.PRUNED for t in study.trials)
early_stopped = sum(
    bool(t.user_attrs.get("early_stopped", False))
    for t in study.trials
    if t.state == optuna.trial.TrialState.COMPLETE
)

print("completed:", completed)
print("pruned:", pruned)
print("early-stopped completed trials:", early_stopped)

importances = get_param_importances(study)
importance_df = pd.DataFrame(
    {"parameter": list(importances.keys()),
     "importance": list(importances.values())}
)
display(importance_df)

# One simple behavior view: mean objective by optimizer and base width.
behavior = (
    trials_df[trials_df["state"] == "COMPLETE"]
    .groupby(["params_optimizer", "params_base_channels"], dropna=False)["value"]
    .agg(["mean", "count"])
    .sort_values("mean", ascending=False)
)
display(behavior)


## B6 — Retrain the best configuration and evaluate the test set once

Now the model-selection process is complete. Rebuild a **fresh** StudentCNN using `study.best_params`, retrain it on the original training split, monitor validation performance, restore the best validation state, and only then evaluate `cifar_test_loader`.

The test result must not be used to go back and alter the search space.


In [ ]:
best_config = dict(study.best_params)
final_model = build_student_model_from_config(best_config)
final_optimizer = make_optimizer(final_model, best_config)
final_criterion = nn.CrossEntropyLoss()

FINAL_MAX_EPOCHS = 8
FINAL_PATIENCE = 3

best_state = None
best_val_acc = 0.0
best_val_loss = float("inf")
epochs_without_improvement = 0

for epoch in range(FINAL_MAX_EPOCHS):
    train_loss, train_acc = train_one_epoch(
        final_model, cifar_train_loader, final_criterion, final_optimizer
    )
    val_loss, val_acc = evaluate(
        final_model, cifar_valid_loader, final_criterion
    )

    print(
        f"final {epoch + 1:02d} | "
        f"train loss {train_loss:.4f} acc {train_acc:.3f} | "
        f"valid loss {val_loss:.4f} acc {val_acc:.3f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_val_loss = val_loss
        best_state = copy.deepcopy(final_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= FINAL_PATIENCE:
        print("final training early-stopped")
        break

final_model.load_state_dict(best_state)
final_test_loss, final_test_acc = evaluate(
    final_model, cifar_test_loader, final_criterion
)
print(f"best validation accuracy: {best_val_acc:.3f}")
print(f"FINAL test accuracy:      {final_test_acc:.3f}")


## B7 — Save and reload the final model

Save enough information to reproduce the model later:
- state dictionary;
- best hyperparameters;
- class names;
- validation/test metrics;
- optional notes about preprocessing.


In [ ]:
CHECKPOINT_PATH = "cifar10_studentcnn_optuna_best.pt"

checkpoint = {
    "model_state_dict": final_model.state_dict(),
    "best_params": best_config,
    "class_names": cifar_classes,
    "best_validation_accuracy": best_val_acc,
    "test_accuracy": final_test_acc,
    "cifar_mean": CIFAR_MEAN,
    "cifar_std": CIFAR_STD,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print("saved:", CHECKPOINT_PATH)

loaded = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
reloaded_model = build_student_model_from_config(loaded["best_params"])
reloaded_model.load_state_dict(loaded["model_state_dict"])
reload_test_loss, reload_test_acc = evaluate(
    reloaded_model, cifar_test_loader, final_criterion
)
print(f"reloaded test accuracy: {reload_test_acc:.3f}")


## B8 — Final comparison and conclusions

Create a compact comparison table using **validation metrics** for model selection:

| Approach | Model | Trainable parameters | Validation accuracy | Notes |
|---|---|---:|---:|---|
| From scratch | Your CNN | ... | ... | ... |
| Transfer learning | frozen backbone | ... | ... | ... |
| Transfer learning | partial fine-tuning | ... | ... | ... |
| Optuna | tuned StudentCNN | ... | ... | ... |

Then report the **single final test result** for the Optuna-selected model.

### Final questions
1. Which approach gave the best validation result, and at what computational cost?
2. What did transfer learning provide that your randomly initialized CNN did not?
3. Which Optuna hyperparameters were most influential in your study?
4. Which interactions between hyperparameters did you observe?
5. Why would a larger number of Optuna trials make your conclusions more reliable?
6. What would you change if accuracy were the only goal? What would you change if latency or memory were also important?


# Instructor notes

## Live Fashion-MNIST section
Emphasize the sequence **pretrained representation → replace head → freeze → train head → selectively unfreeze**. The important result is conceptual, not the absolute accuracy. Fashion-MNIST is deliberately a domain-mismatch case relative to ImageNet.

A useful advanced remark: `requires_grad=False` freezes parameters, but calling `model.train()` can still update BatchNorm running statistics. For a short introductory transfer-learning demonstration this nuance can be mentioned without changing the training helper; in production experiments, decide explicitly whether frozen BatchNorm statistics should also remain fixed.

## CIFAR-10 student CNN
The reference solution uses `AdaptiveAvgPool2d(1)` so the classifier does not depend on a hand-computed flattened spatial size. Students do **not** need to reproduce this architecture; assess whether their design is internally consistent and whether they can justify it.

## Transfer-learning comparison
CIFAR-10 is closer to ImageNet than Fashion-MNIST because both contain natural RGB imagery, although the 32×32 resolution is much smaller. The default weight transforms resize CIFAR images substantially, so discuss the accuracy/compute trade-off.

## Optuna
The reference search is intentionally small. Students should not claim that 12 trials establish globally optimal hyperparameters. More important is that they:
- isolate the test set from tuning;
- understand conditional parameters such as SGD momentum;
- distinguish Optuna pruning from per-trial early stopping;
- analyze patterns rather than reporting only the single best trial;
- retrain a fresh model with the selected configuration.
